# Deep Learning Mini-Project: LSTM Models for Text Generation, Translation, and Classification

This notebook implements three recurrent neural network applications with TensorFlow/Keras:

1. Character-level text autocomplete using a two-layer LSTM trained on *Frankenstein*.
2. English-to-French neural machine translation using a Seq2Seq encoder-decoder LSTM.
3. AG News multi-class text classification using LSTM-based sequence models.

The notebook is organized for direct academic submission. Each part includes preprocessing, model construction, training, evaluation, visualizations, sample predictions, and a short analysis.

## Dataset Download and Execution Instructions

Run this notebook from top to bottom in a Python environment with TensorFlow installed.

Datasets are downloaded automatically:

- *Frankenstein* is downloaded from Project Gutenberg through `tf.keras.utils.get_file`.
- The English-French dataset is downloaded from the ManyThings/Tatoeba Anki collection and limited to 10,000 sentence pairs.
- AG News is downloaded through `tensorflow_datasets`.

Recommended execution environment:

- Python 3.10 or newer.
- TensorFlow 2.15 or newer.
- A GPU is recommended for the full training configuration.

The first LSTM text generation model is intentionally trained for 32 epochs as required. On CPU this can take a long time.

In [ ]:
# Global imports and reproducibility setup
import os
import re
import random
import string
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## Shared Helper Functions

These utility functions keep plotting, text cleaning, and prediction display consistent across the three parts.

In [ ]:
def plot_training_curves(history, metrics=("loss", "accuracy"), title_prefix="Model"):
    'Plot training and validation curves from a Keras History object.'
    history_df = pd.DataFrame(history.history)
    available_metrics = [m for m in metrics if m in history_df.columns]
    if not available_metrics:
        available_metrics = ["loss"]

    fig, axes = plt.subplots(1, len(available_metrics), figsize=(6 * len(available_metrics), 4))
    if len(available_metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, available_metrics):
        ax.plot(history_df[metric], label=f"train_{metric}")
        val_metric = f"val_{metric}"
        if val_metric in history_df:
            ax.plot(history_df[val_metric], label=val_metric)
        ax.set_title(f"{title_prefix}: {metric}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric)
        ax.legend()

    plt.tight_layout()
    plt.show()


def normalize_spaces(text):
    'Lowercase text and collapse repeated whitespace.'
    return re.sub(r"\s+", " ", text.lower()).strip()


def clean_sentence(text):
    'Basic sentence cleaning used for translation and classification examples.'
    text = str(text).lower().strip()
    text = re.sub(r"[^a-zA-Z\u00C0-\u00FF?.!,;:'\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Part 1 - Text Autocomplete with LSTM

## Objective

The first model is an autoregressive character-level language model. Given a fixed-length sequence of characters, it predicts the next character. During generation, the predicted character is appended to the context and fed back into the model.

This is a useful educational setup because it exposes the complete sequence modeling pipeline: character encoding, sliding-window sample creation, recurrent modeling, and autoregressive decoding.

## 1.1 Text Preprocessing

The source text is downloaded from Project Gutenberg, converted to lowercase, and normalized by collapsing repeated whitespace. Gutenberg header/footer markers are removed when present so the model focuses on the novel text.

In [ ]:
FRANKENSTEIN_URL = "https://www.gutenberg.org/files/84/84-0.txt"
frankenstein_path = keras.utils.get_file("frankenstein_gutenberg_84.txt", FRANKENSTEIN_URL)

raw_text = Path(frankenstein_path).read_text(encoding="utf-8")

start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN"

start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)
if start_idx != -1 and end_idx != -1:
    raw_text = raw_text[start_idx:end_idx]

text = normalize_spaces(raw_text)

print("Number of characters after cleaning:", len(text))
print("Preview:")
print(text[:600])

## 1.2 Character Encoding

Each unique character becomes a vocabulary item. We create two dictionaries:

- `char_to_idx`: maps a character to an integer id.
- `idx_to_char`: maps an integer id back to a character.

The vocabulary size determines the width of the one-hot representation and the number of classes in the softmax output layer.

In [ ]:
chars = sorted(set(text))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print("Characters:")
print("".join(chars))

## 1.3 Sliding-Window Training Sample Generation

The model receives a sequence of `SEQ_LENGTH` characters and learns to predict the next character. A sliding window creates many overlapping training examples.

For memory efficiency, integer sequences are stored first. One-hot encoding is applied inside a `tf.data` pipeline before training, which still satisfies the one-hot input requirement while avoiding a very large dense array in RAM.

In [ ]:
SEQ_LENGTH = 60
STEP = 3
BATCH_SIZE_CHAR = 256

sentences = []
next_chars = []

for i in range(0, len(text) - SEQ_LENGTH, STEP):
    sentences.append(text[i : i + SEQ_LENGTH])
    next_chars.append(text[i + SEQ_LENGTH])

x_char_int = np.zeros((len(sentences), SEQ_LENGTH), dtype=np.int32)
y_char_int = np.zeros((len(sentences),), dtype=np.int32)

for i, sentence in enumerate(sentences):
    x_char_int[i] = [char_to_idx[ch] for ch in sentence]
    y_char_int[i] = char_to_idx[next_chars[i]]

print("Number of training samples:", len(sentences))
print("Input integer tensor shape:", x_char_int.shape)
print("Target integer tensor shape:", y_char_int.shape)

validation_fraction = 0.1
num_val = int(len(x_char_int) * validation_fraction)
x_train_char, x_val_char = x_char_int[:-num_val], x_char_int[-num_val:]
y_train_char, y_val_char = y_char_int[:-num_val], y_char_int[-num_val:]


def one_hot_char_batch(x, y):
    x = tf.one_hot(x, depth=vocab_size)
    y = tf.one_hot(y, depth=vocab_size)
    return x, y


train_char_ds = (
    tf.data.Dataset.from_tensor_slices((x_train_char, y_train_char))
    .shuffle(20_000, seed=SEED)
    .batch(BATCH_SIZE_CHAR)
    .map(one_hot_char_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_char_ds = (
    tf.data.Dataset.from_tensor_slices((x_val_char, y_val_char))
    .batch(BATCH_SIZE_CHAR)
    .map(one_hot_char_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

## 1.4 Model Architecture and Hyperparameters

The required model uses two LSTM layers. The first LSTM returns the full sequence so the second LSTM can process temporal features at every position. Dropout and recurrent dropout reduce overfitting.

Key hyperparameters:

- `SEQ_LENGTH = 60`: context length used to predict the next character.
- `BATCH_SIZE = 256`: stable gradient estimate while remaining practical on most GPUs.
- `EPOCHS = 32`: required full training duration.
- `DROPOUT = 0.2` and `RECURRENT_DROPOUT = 0.2`: regularization for input and recurrent connections.

In [ ]:
char_model = keras.Sequential(
    [
        layers.Input(shape=(SEQ_LENGTH, vocab_size)),
        layers.LSTM(
            256,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2,
            name="lstm_1",
        ),
        layers.LSTM(
            256,
            dropout=0.2,
            recurrent_dropout=0.2,
            name="lstm_2",
        ),
        layers.Dense(vocab_size, activation="softmax", name="next_character_softmax"),
    ],
    name="frankenstein_char_lstm",
)

char_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

char_model.summary()

## 1.5 Training

The model is trained with a chronological validation split. This evaluates generalization on later parts of the novel rather than on randomly mixed fragments.

In [ ]:
EPOCHS_CHAR = 32

char_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_frankenstein_char_lstm.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
    ),
]

char_history = char_model.fit(
    train_char_ds,
    validation_data=val_char_ds,
    epochs=EPOCHS_CHAR,
    callbacks=char_callbacks,
)

## 1.6 Evaluation and Loss Visualization

Training and validation loss show whether the language model is learning useful next-character distributions and whether it is overfitting.

In [ ]:
plot_training_curves(char_history, metrics=("loss", "accuracy"), title_prefix="Character LSTM")

## 1.7 Text Generation: Greedy Decoding and Beam Search

Greedy decoding chooses the most likely next character at every step. Beam search keeps multiple candidate continuations and often produces more coherent text because it considers sequence-level probability.

In [ ]:
def prepare_prompt(prompt, seq_length=SEQ_LENGTH):
    prompt = normalize_spaces(prompt)
    prompt = "".join(ch for ch in prompt if ch in char_to_idx)
    if len(prompt) < seq_length:
        prompt = (" " * (seq_length - len(prompt))) + prompt
    return prompt[-seq_length:]


def predict_next_char_distribution(model, context):
    encoded = np.array([[char_to_idx.get(ch, char_to_idx[" "]) for ch in context]], dtype=np.int32)
    one_hot = tf.one_hot(encoded, depth=vocab_size)
    probs = model.predict(one_hot, verbose=0)[0]
    return probs


def generate_greedy(model, prompt, length=200):
    context = prepare_prompt(prompt)
    generated = prompt

    for _ in range(length):
        probs = predict_next_char_distribution(model, context)
        next_idx = int(np.argmax(probs))
        next_char = idx_to_char[next_idx]
        generated += next_char
        context = (context + next_char)[-SEQ_LENGTH:]

    return generated


def generate_beam_search(model, prompt, length=200, beam_width=3):
    initial_context = prepare_prompt(prompt)
    beams = [(initial_context, prompt, 0.0)]

    for _ in range(length):
        candidates = []
        for context, generated, score in beams:
            probs = predict_next_char_distribution(model, context)
            top_indices = np.argsort(probs)[-beam_width:]
            for idx in top_indices:
                ch = idx_to_char[int(idx)]
                new_context = (context + ch)[-SEQ_LENGTH:]
                new_score = score + np.log(float(probs[idx]) + 1e-12)
                candidates.append((new_context, generated + ch, new_score))
        beams = sorted(candidates, key=lambda item: item[2], reverse=True)[:beam_width]

    return beams[0][1]


prompt = "it was on a dreary night of november"
print("Prompt:", prompt)
print("\nGreedy generation:\n")
print(generate_greedy(char_model, prompt, length=250))
print("\nBeam-search generation:\n")
print(generate_beam_search(char_model, prompt, length=250, beam_width=3))

## Part 1 Notes

Greedy decoding is faster and deterministic, but it may repeat common character patterns. Beam search is slower because it evaluates several candidate continuations, but it can produce text with better local consistency. Character-level generation usually needs many epochs and large text corpora to produce fluent paragraphs.

# Part 2 - English to French Translation with Seq2Seq LSTM

## Objective

This part builds a neural machine translation model with an encoder-decoder LSTM architecture. The encoder reads an English sentence and compresses it into recurrent state vectors. The decoder uses teacher forcing during training to predict the French sentence one token at a time.

## 2.1 Data Loading and Cleaning

The ManyThings English-French sentence-pair dataset is downloaded and limited to a maximum of 10,000 pairs as required. Missing rows are removed. Text is lowercased and lightly cleaned.

In [ ]:
FRA_ENG_URL = "http://www.manythings.org/anki/fra-eng.zip"
fra_zip_path = keras.utils.get_file("fra-eng.zip", FRA_ENG_URL, extract=False)

with zipfile.ZipFile(fra_zip_path, "r") as zf:
    zf.extractall(Path(fra_zip_path).parent / "fra-eng")

fra_txt_path = Path(fra_zip_path).parent / "fra-eng" / "fra.txt"
raw_pairs = pd.read_csv(
    fra_txt_path,
    sep="\t",
    header=None,
    names=["english", "french", "metadata"],
    usecols=[0, 1, 2],
)

pairs = raw_pairs[["english", "french"]].dropna().head(10_000).copy()
pairs["english"] = pairs["english"].map(clean_sentence)
pairs["french"] = pairs["french"].map(clean_sentence)
pairs = pairs[(pairs["english"].str.len() > 0) & (pairs["french"].str.len() > 0)].reset_index(drop=True)

pairs["french_in"] = "<start> " + pairs["french"]
pairs["french_out"] = pairs["french"] + " <end>"

print("Sentence pairs:", len(pairs))
pairs.head()

## 2.2 Tokenization and Vocabulary Creation

Separate tokenizers are used for English and French. French decoder sequences include `<start>` and `<end>` tokens, which mark the beginning and end of generated translations.

In [ ]:
MAX_PAIRS = 10_000
MAX_NUM_WORDS = 12_000

eng_tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_NUM_WORDS,
    filters="",
    lower=False,
    oov_token="<unk>",
)
fra_tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_NUM_WORDS,
    filters="",
    lower=False,
    oov_token="<unk>",
)

eng_tokenizer.fit_on_texts(pairs["english"])
fra_tokenizer.fit_on_texts(pd.concat([pairs["french_in"], pairs["french_out"]]))

eng_vocab_size = min(MAX_NUM_WORDS, len(eng_tokenizer.word_index) + 1)
fra_vocab_size = min(MAX_NUM_WORDS, len(fra_tokenizer.word_index) + 1)

print("English vocabulary size:", eng_vocab_size)
print("French vocabulary size:", fra_vocab_size)
print("French special token ids:", {
    "<start>": fra_tokenizer.word_index.get("<start>"),
    "<end>": fra_tokenizer.word_index.get("<end>"),
})

## 2.3 Sequence Preparation with Teacher Forcing

Teacher forcing feeds the correct previous French token into the decoder during training. Therefore:

- Encoder input: English sentence.
- Decoder input: French sentence starting with `<start>`.
- Decoder target: French sentence ending with `<end>`.

All sequences are padded to fixed lengths.

In [ ]:
encoder_sequences = eng_tokenizer.texts_to_sequences(pairs["english"])
decoder_input_sequences = fra_tokenizer.texts_to_sequences(pairs["french_in"])
decoder_target_sequences = fra_tokenizer.texts_to_sequences(pairs["french_out"])

max_encoder_len = max(len(seq) for seq in encoder_sequences)
max_decoder_len = max(len(seq) for seq in decoder_input_sequences)

encoder_input_data = keras.preprocessing.sequence.pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding="post",
)
decoder_input_data = keras.preprocessing.sequence.pad_sequences(
    decoder_input_sequences,
    maxlen=max_decoder_len,
    padding="post",
)
decoder_target_data = keras.preprocessing.sequence.pad_sequences(
    decoder_target_sequences,
    maxlen=max_decoder_len,
    padding="post",
)
decoder_target_data = np.expand_dims(decoder_target_data, axis=-1)

split_idx = int(len(encoder_input_data) * 0.9)

train_inputs = [encoder_input_data[:split_idx], decoder_input_data[:split_idx]]
val_inputs = [encoder_input_data[split_idx:], decoder_input_data[split_idx:]]
y_train_trans = decoder_target_data[:split_idx]
y_val_trans = decoder_target_data[split_idx:]

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)

## 2.4 Seq2Seq Model Architecture

The encoder consists of an embedding layer followed by an LSTM that returns the final hidden and cell states. The decoder has its own embedding layer and LSTM. The decoder LSTM is initialized with the encoder states, then a dense softmax layer predicts the next French token at each time step.

In [ ]:
EMBEDDING_DIM = 128
LSTM_UNITS = 256
BATCH_SIZE_TRANS = 64
EPOCHS_TRANS = 20

encoder_inputs = keras.Input(shape=(None,), name="encoder_inputs")
encoder_embedding = layers.Embedding(
    eng_vocab_size,
    EMBEDDING_DIM,
    mask_zero=True,
    name="encoder_embedding",
)(encoder_inputs)
_, state_h, state_c = layers.LSTM(
    LSTM_UNITS,
    return_state=True,
    name="encoder_lstm",
)(encoder_embedding)
encoder_states = [state_h, state_c]

decoder_inputs = keras.Input(shape=(None,), name="decoder_inputs")
decoder_embedding_layer = layers.Embedding(
    fra_vocab_size,
    EMBEDDING_DIM,
    mask_zero=True,
    name="decoder_embedding",
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)
decoder_lstm = layers.LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm",
)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = layers.Dense(fra_vocab_size, activation="softmax", name="decoder_softmax")
decoder_outputs = decoder_dense(decoder_outputs)

translation_model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs, name="eng_fra_seq2seq_lstm")

translation_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

translation_model.summary()

## 2.5 Training

The model is optimized with Adam and sparse categorical cross-entropy. Sparse targets avoid building a very large one-hot target tensor.

In [ ]:
translation_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_eng_fra_seq2seq_lstm.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
]

translation_history = translation_model.fit(
    train_inputs,
    y_train_trans,
    validation_data=(val_inputs, y_val_trans),
    batch_size=BATCH_SIZE_TRANS,
    epochs=EPOCHS_TRANS,
    callbacks=translation_callbacks,
)

## 2.6 Training Visualization

Loss and accuracy curves help diagnose underfitting, overfitting, and convergence behavior.

In [ ]:
plot_training_curves(translation_history, metrics=("loss", "accuracy"), title_prefix="Seq2Seq Translation")

## 2.7 Inference Models and Custom Translation

Training uses full target sequences, while inference generates one token at a time. Therefore, separate inference models are built for the encoder and decoder.

In [ ]:
# Encoder inference model: English sequence -> final LSTM states
encoder_model = keras.Model(encoder_inputs, encoder_states, name="encoder_inference_model")

# Decoder inference model: previous French token + previous states -> next token + new states
decoder_state_input_h = keras.Input(shape=(LSTM_UNITS,), name="decoder_state_input_h")
decoder_state_input_c = keras.Input(shape=(LSTM_UNITS,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_single_embedding = decoder_embedding_layer(decoder_inputs)
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_single_embedding,
    initial_state=decoder_states_inputs,
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf,
    name="decoder_inference_model",
)

fra_index_to_word = {idx: word for word, idx in fra_tokenizer.word_index.items()}
start_token_id = fra_tokenizer.word_index["<start>"]
end_token_id = fra_tokenizer.word_index["<end>"]


def translate_sentence(sentence, max_len=max_decoder_len):
    sentence = clean_sentence(sentence)
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = keras.preprocessing.sequence.pad_sequences(seq, maxlen=max_encoder_len, padding="post")

    states = encoder_model.predict(seq, verbose=0)
    target_seq = np.array([[start_token_id]], dtype=np.int32)

    decoded_tokens = []
    for _ in range(max_len):
        output_tokens, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        sampled_token_id = int(np.argmax(output_tokens[0, -1, :]))

        if sampled_token_id == end_token_id or sampled_token_id == 0:
            break

        sampled_word = fra_index_to_word.get(sampled_token_id, "<unk>")
        decoded_tokens.append(sampled_word)

        target_seq = np.array([[sampled_token_id]], dtype=np.int32)
        states = [h, c]

    return " ".join(decoded_tokens)


for idx in [0, 10, 100, 500, 1000]:
    source = pairs.loc[idx, "english"]
    expected = pairs.loc[idx, "french"]
    predicted = translate_sentence(source)
    print(f"EN: {source}")
    print(f"FR expected:  {expected}")
    print(f"FR predicted: {predicted}")
    print("-" * 80)

custom_sentences = [
    "i am happy",
    "where is the station?",
    "this book is interesting",
    "we are learning deep learning",
]

for sentence in custom_sentences:
    print(f"{sentence} -> {translate_sentence(sentence)}")

## 2.8 Error Analysis

Typical errors in this compact translation setting include:

- Short generic outputs for rare or long English inputs.
- Incorrect gender or number agreement in French.
- Missing punctuation.
- Poor handling of words outside the 10,000-pair training subset.

These issues are expected because the dataset subset is small and the model does not use attention. An attention-based encoder-decoder or Transformer would usually perform better.

## 2.9 Experiments: Embedding Size, LSTM Units, GRU, and Bidirectional Encoder

The following helper functions allow controlled experiments with model size and recurrent cell type. To keep the notebook practical, the default experiment training uses fewer epochs than the main model. Increase `EXPERIMENT_EPOCHS` for a stronger comparison.

In [ ]:
def build_seq2seq_variant(embedding_dim=64, units=128, cell_type="lstm", bidirectional_encoder=False):
    enc_inputs = keras.Input(shape=(None,), name=f"{cell_type}_enc_inputs")
    enc_emb = layers.Embedding(eng_vocab_size, embedding_dim, mask_zero=True)(enc_inputs)

    if cell_type == "gru":
        if bidirectional_encoder:
            enc_out = layers.Bidirectional(layers.GRU(units, return_state=True))
            _, forward_h, backward_h = enc_out(enc_emb)
            enc_state = [layers.Concatenate()([forward_h, backward_h])]
            dec_units = units * 2
        else:
            _, h = layers.GRU(units, return_state=True)(enc_emb)
            enc_state = [h]
            dec_units = units

        dec_inputs = keras.Input(shape=(None,), name=f"{cell_type}_dec_inputs")
        dec_emb = layers.Embedding(fra_vocab_size, embedding_dim, mask_zero=True)(dec_inputs)
        dec_gru = layers.GRU(dec_units, return_sequences=True, return_state=True)
        dec_outputs, _ = dec_gru(dec_emb, initial_state=enc_state)
    else:
        if bidirectional_encoder:
            enc_bi = layers.Bidirectional(layers.LSTM(units, return_state=True))
            _, forward_h, forward_c, backward_h, backward_c = enc_bi(enc_emb)
            h = layers.Concatenate()([forward_h, backward_h])
            c = layers.Concatenate()([forward_c, backward_c])
            enc_state = [h, c]
            dec_units = units * 2
        else:
            _, h, c = layers.LSTM(units, return_state=True)(enc_emb)
            enc_state = [h, c]
            dec_units = units

        dec_inputs = keras.Input(shape=(None,), name=f"{cell_type}_dec_inputs")
        dec_emb = layers.Embedding(fra_vocab_size, embedding_dim, mask_zero=True)(dec_inputs)
        dec_lstm = layers.LSTM(dec_units, return_sequences=True, return_state=True)
        dec_outputs, _, _ = dec_lstm(dec_emb, initial_state=enc_state)

    dec_outputs = layers.Dense(fra_vocab_size, activation="softmax")(dec_outputs)
    model = keras.Model([enc_inputs, dec_inputs], dec_outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


EXPERIMENT_EPOCHS = 3
experiment_configs = [
    {"name": "LSTM small embedding", "embedding_dim": 64, "units": 128, "cell_type": "lstm", "bidirectional_encoder": False},
    {"name": "LSTM larger units", "embedding_dim": 128, "units": 256, "cell_type": "lstm", "bidirectional_encoder": False},
    {"name": "GRU encoder-decoder", "embedding_dim": 128, "units": 256, "cell_type": "gru", "bidirectional_encoder": False},
    {"name": "Bidirectional LSTM encoder", "embedding_dim": 128, "units": 128, "cell_type": "lstm", "bidirectional_encoder": True},
]

experiment_results = []

for config in experiment_configs:
    print("Running:", config["name"])
    variant = build_seq2seq_variant(
        embedding_dim=config["embedding_dim"],
        units=config["units"],
        cell_type=config["cell_type"],
        bidirectional_encoder=config["bidirectional_encoder"],
    )
    hist = variant.fit(
        train_inputs,
        y_train_trans,
        validation_data=(val_inputs, y_val_trans),
        batch_size=BATCH_SIZE_TRANS,
        epochs=EXPERIMENT_EPOCHS,
        verbose=1,
    )
    experiment_results.append(
        {
            "model": config["name"],
            "final_val_loss": hist.history["val_loss"][-1],
            "final_val_accuracy": hist.history["val_accuracy"][-1],
        }
    )

pd.DataFrame(experiment_results).sort_values("final_val_loss")

# Part 3 - News Classification with LSTM

## Objective

This part builds a multi-class classifier for AG News. Each news headline/body text is converted into a padded integer sequence and classified into one of four categories:

1. World
2. Sports
3. Business
4. Sci/Tech

## 3.1 Dataset Loading

AG News is loaded with `tensorflow_datasets`. The official train/test split is used, and part of the training split is reserved for validation.

In [ ]:
import tensorflow_datasets as tfds

ag_train_raw, ag_test_raw = tfds.load(
    "ag_news_subset",
    split=["train", "test"],
    as_supervised=True,
)

class_names = ["World", "Sports", "Business", "Sci/Tech"]

train_texts = []
train_labels = []
test_texts = []
test_labels = []

for text_tensor, label_tensor in tfds.as_numpy(ag_train_raw):
    train_texts.append(text_tensor.decode("utf-8"))
    train_labels.append(int(label_tensor))

for text_tensor, label_tensor in tfds.as_numpy(ag_test_raw):
    test_texts.append(text_tensor.decode("utf-8"))
    test_labels.append(int(label_tensor))

train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
test_df = pd.DataFrame({"text": test_texts, "label": test_labels})

print("Train size:", len(train_df))
print("Test size:", len(test_df))
train_df.head()

## 3.2 Text Preprocessing, Tokenization, Vocabulary, and Padding

Text is lowercased and cleaned. A Keras tokenizer builds the vocabulary from training text only. Sentences are converted to integer sequences and padded to a fixed length.

In [ ]:
def clean_news_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s.,!?'-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


MAX_NEWS_WORDS = 30_000
MAX_NEWS_LEN = 120
BATCH_SIZE_NEWS = 128

train_df["clean_text"] = train_df["text"].map(clean_news_text)
test_df["clean_text"] = test_df["text"].map(clean_news_text)

news_tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_NEWS_WORDS,
    oov_token="<unk>",
)
news_tokenizer.fit_on_texts(train_df["clean_text"])
news_vocab_size = min(MAX_NEWS_WORDS, len(news_tokenizer.word_index) + 1)

x_news_all = keras.preprocessing.sequence.pad_sequences(
    news_tokenizer.texts_to_sequences(train_df["clean_text"]),
    maxlen=MAX_NEWS_LEN,
    padding="post",
    truncating="post",
)
y_news_all = train_df["label"].to_numpy()

x_news_test = keras.preprocessing.sequence.pad_sequences(
    news_tokenizer.texts_to_sequences(test_df["clean_text"]),
    maxlen=MAX_NEWS_LEN,
    padding="post",
    truncating="post",
)
y_news_test = test_df["label"].to_numpy()

val_size = int(0.1 * len(x_news_all))
x_news_train, x_news_val = x_news_all[:-val_size], x_news_all[-val_size:]
y_news_train, y_news_val = y_news_all[:-val_size], y_news_all[-val_size:]

print("Vocabulary size:", news_vocab_size)
print("Train shape:", x_news_train.shape)
print("Validation shape:", x_news_val.shape)
print("Test shape:", x_news_test.shape)

## 3.3 LSTM Classifier Architecture

The classifier uses an embedding layer, a bidirectional LSTM for stronger context modeling, dropout regularization, and a softmax classifier. Bidirectionality is helpful for classification because the whole text is available before prediction.

In [ ]:
NEWS_EMBEDDING_DIM = 128
NEWS_LSTM_UNITS = 128
NEWS_EPOCHS = 6

news_model = keras.Sequential(
    [
        layers.Input(shape=(MAX_NEWS_LEN,)),
        layers.Embedding(news_vocab_size, NEWS_EMBEDDING_DIM, mask_zero=True),
        layers.Bidirectional(layers.LSTM(NEWS_LSTM_UNITS, return_sequences=False)),
        layers.Dropout(0.4),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(len(class_names), activation="softmax"),
    ],
    name="ag_news_bilstm_classifier",
)

news_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

news_model.summary()

## 3.4 Training

The model is trained on the training split and validated on a held-out validation split. Early stopping restores the best validation weights.

In [ ]:
news_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_ag_news_lstm.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=2,
        restore_best_weights=True,
    ),
]

news_history = news_model.fit(
    x_news_train,
    y_news_train,
    validation_data=(x_news_val, y_news_val),
    batch_size=BATCH_SIZE_NEWS,
    epochs=NEWS_EPOCHS,
    callbacks=news_callbacks,
)

## 3.5 Training Curves

Accuracy and loss curves summarize optimization progress and validation behavior.

In [ ]:
plot_training_curves(news_history, metrics=("loss", "accuracy"), title_prefix="AG News BiLSTM")

## 3.6 Evaluation: Accuracy, Classification Report, and Confusion Matrix

The test set evaluates final generalization. A classification report shows class-level precision, recall, and F1-score. The confusion matrix shows which categories are commonly confused.

In [ ]:
test_loss, test_accuracy = news_model.evaluate(x_news_test, y_news_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_news_prob = news_model.predict(x_news_test, batch_size=BATCH_SIZE_NEWS, verbose=0)
y_news_pred = np.argmax(y_news_prob, axis=1)

print(classification_report(y_news_test, y_news_pred, target_names=class_names))

cm = confusion_matrix(y_news_test, y_news_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("AG News Confusion Matrix")
plt.tight_layout()
plt.show()

## 3.7 Error Analysis

Misclassified examples help identify whether the classifier struggles with ambiguous wording, short texts, or categories with overlapping vocabulary.

In [ ]:
errors = np.where(y_news_pred != y_news_test)[0]
print("Number of test errors:", len(errors))

for idx in errors[:10]:
    print("Text:", test_df.iloc[idx]["text"])
    print("Expected:", class_names[y_news_test[idx]])
    print("Predicted:", class_names[y_news_pred[idx]])
    print("-" * 80)

## 3.8 Testing on Custom News Headlines

The trained model can predict labels for new custom headlines.

In [ ]:
def predict_news_category(texts):
    cleaned = [clean_news_text(t) for t in texts]
    seqs = news_tokenizer.texts_to_sequences(cleaned)
    padded = keras.preprocessing.sequence.pad_sequences(
        seqs,
        maxlen=MAX_NEWS_LEN,
        padding="post",
        truncating="post",
    )
    probs = news_model.predict(padded, verbose=0)
    labels = np.argmax(probs, axis=1)

    results = []
    for text, label, prob in zip(texts, labels, probs):
        results.append(
            {
                "headline": text,
                "predicted_category": class_names[int(label)],
                "confidence": float(np.max(prob)),
            }
        )
    return pd.DataFrame(results)


custom_headlines = [
    "Global leaders meet to discuss a new climate agreement",
    "The national team wins the championship after a dramatic final",
    "Technology stocks rise as chip demand increases",
    "A new spacecraft sends detailed images from Mars",
]

predict_news_category(custom_headlines)

## 3.9 Bonus Experiments: Dropout, GRU, and Bidirectional LSTM

The main classifier already includes dropout and a bidirectional LSTM. The following compact experiment compares LSTM, GRU, and bidirectional LSTM variants under the same preprocessing setup.

In [ ]:
def build_news_variant(cell="lstm", bidirectional=False, dropout_rate=0.3):
    inputs = keras.Input(shape=(MAX_NEWS_LEN,))
    x = layers.Embedding(news_vocab_size, NEWS_EMBEDDING_DIM, mask_zero=True)(inputs)

    if cell == "gru":
        recurrent_layer = layers.GRU(NEWS_LSTM_UNITS)
    else:
        recurrent_layer = layers.LSTM(NEWS_LSTM_UNITS)

    if bidirectional:
        x = layers.Bidirectional(recurrent_layer)(x)
    else:
        x = recurrent_layer(x)

    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model


NEWS_EXPERIMENT_EPOCHS = 2
news_experiment_configs = [
    {"name": "LSTM + dropout", "cell": "lstm", "bidirectional": False, "dropout_rate": 0.3},
    {"name": "GRU + dropout", "cell": "gru", "bidirectional": False, "dropout_rate": 0.3},
    {"name": "Bidirectional LSTM + dropout", "cell": "lstm", "bidirectional": True, "dropout_rate": 0.3},
]

news_experiment_results = []

for cfg in news_experiment_configs:
    print("Running:", cfg["name"])
    model = build_news_variant(
        cell=cfg["cell"],
        bidirectional=cfg["bidirectional"],
        dropout_rate=cfg["dropout_rate"],
    )
    hist = model.fit(
        x_news_train,
        y_news_train,
        validation_data=(x_news_val, y_news_val),
        batch_size=BATCH_SIZE_NEWS,
        epochs=NEWS_EXPERIMENT_EPOCHS,
        verbose=1,
    )
    news_experiment_results.append(
        {
            "model": cfg["name"],
            "final_val_loss": hist.history["val_loss"][-1],
            "final_val_accuracy": hist.history["val_accuracy"][-1],
        }
    )

pd.DataFrame(news_experiment_results).sort_values("final_val_accuracy", ascending=False)

# Final Technical Report

## Architecture Choices

**Part 1: Character autocomplete.** A two-layer LSTM is used because character generation requires temporal memory over dozens of characters. The first LSTM returns sequences so the second layer can model higher-level temporal patterns. A softmax layer predicts the next character.

**Part 2: Translation.** The translation model uses a classic encoder-decoder Seq2Seq architecture. The encoder LSTM compresses the English input into hidden and cell states. The decoder LSTM receives the previous French token and predicts the next token using teacher forcing.

**Part 3: News classification.** The classifier uses an embedding layer and a bidirectional LSTM. Bidirectionality is appropriate because classification has access to the full input sequence, unlike autoregressive generation.

## Hyperparameters

| Part | Main hyperparameters |
|---|---|
| Character LSTM | sequence length 60, batch size 256, 2 LSTM layers, 256 units each, dropout 0.2, recurrent dropout 0.2, 32 epochs |
| Seq2Seq translation | 10,000 sentence pairs, embedding size 128, LSTM units 256, batch size 64, 20 epochs |
| AG News classification | vocabulary 30,000, max length 120, embedding size 128, LSTM units 128, batch size 128, 6 epochs |

## Training Configuration

All models use TensorFlow/Keras and Adam optimization. The character model uses categorical cross-entropy because the next-character target is one-hot encoded. The translation and classification models use sparse categorical cross-entropy to avoid very large dense target matrices.

## Results

Results should be filled after execution using the plotted curves and printed metrics:

- Character LSTM: report final training/validation loss and inspect generated samples.
- Translation: report validation loss/accuracy and compare predicted translations with expected French sentences.
- AG News: report test accuracy, classification report, confusion matrix, and observed error patterns.

## Limitations

- The character model is trained at character level, which makes long-range semantic consistency difficult.
- The translation model uses only 10,000 sentence pairs and no attention mechanism, so longer sentences and rare words are difficult.
- AG News texts can be ambiguous, especially between Business and Sci/Tech when articles discuss technology companies.
- Training time can be significant on CPU.

## Future Improvements

- Add temperature sampling to the character generator for more diverse text.
- Add attention or replace the Seq2Seq model with a Transformer for translation.
- Use pretrained embeddings or transformer encoders for AG News classification.
- Perform systematic hyperparameter search and report averaged results over multiple seeds.
- Add BLEU score for translation and save all trained models with versioned experiment names.

# Conclusion

This mini-project demonstrates three core LSTM applications in natural language processing: autoregressive text generation, sequence-to-sequence translation, and multi-class text classification. The implementations include complete preprocessing, model training, evaluation, visualization, and testing workflows. The experiments also show how recurrent architecture choices such as LSTM, GRU, dropout, and bidirectionality affect different NLP tasks.

# `requirements.txt` Section

```text
tensorflow>=2.15
tensorflow-datasets>=4.9
numpy>=1.24
pandas>=2.0
matplotlib>=3.7
seaborn>=0.13
scikit-learn>=1.3
jupyter>=1.0
```

# Project Folder Structure

```text
deep-learning-lstm-mini-project/
├── lstm_nlp_mini_project.ipynb
├── requirements.txt
├── best_frankenstein_char_lstm.keras
├── best_eng_fra_seq2seq_lstm.keras
└── best_ag_news_lstm.keras
```

The `.keras` model files are created automatically after running the training cells with model checkpoints enabled.